<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/mobile_phone_price_range_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "train.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "iabhishekofficial/mobile-price-classification",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_522/4100854409.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'mobile-price-classification' dataset.
First 5 records:    battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \
0        188        2  ...         20       756  2549     9     7         19   
1        136        3  ...        905      1988  2631    17     3          7   
2        145        5  ...       1263      1716  2603    11     2          9   
3        131        6  ...       1216      1786  2769    16     8         11   
4        141        

In [37]:
df

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,794,1,0.5,1,0,1,2,0.8,106,6,...,1222,1890,668,13,4,19,1,1,0,0
1996,1965,1,2.6,1,0,0,39,0.2,187,4,...,915,1965,2032,11,10,16,1,1,1,2
1997,1911,0,0.9,1,1,1,36,0.7,108,8,...,868,1632,3057,9,1,5,1,1,0,3
1998,1512,0,0.9,0,4,1,46,0.1,145,5,...,336,670,869,18,10,19,1,1,1,0


In [38]:
df.columns

Index(['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g',
       'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height',
       'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g',
       'touch_screen', 'wifi', 'price_range'],
      dtype='object')

In [39]:
x = df.drop('price_range',axis=1)
y = df['price_range']

In [40]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x = sc.fit_transform(x)

In [41]:
y.values

array([1, 2, 2, ..., 3, 0, 3])

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset

In [43]:
class MobileDataset(Dataset):
  def __init__(self,x,y):
    self.x = torch.tensor(x,dtype=torch.float32)
    self.y = torch.tensor(y.values,dtype=torch.long)

  def __len__(self):
    return len(self.y)

  def __getitem__(self,idx):
    return self.x[idx],self.y[idx]



In [44]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [45]:
train_loader = DataLoader(MobileDataset(x_train,y_train),batch_size=32,shuffle=True)
test_loader = DataLoader(MobileDataset(x_test,y_test),batch_size=32,shuffle=False)

In [46]:
input_size = x_train.shape[1]
output_size = df['price_range'].unique().shape[0]

print(input_size)
print(output_size)

20
4


In [47]:
model = nn.Sequential(
    nn.Linear(input_size,512),
    nn.ReLU(),
    nn.Linear(512,512),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(512,output_size)
)

In [48]:
optimizer = optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

In [49]:
epochs=20

for epoch in range(epochs):
  total_loss = 0
  for x,y in train_loader:
    optimizer.zero_grad()
    y_pred = model(x)
    loss = criterion(y_pred,y)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}")



Epoch 1/20, Loss: 0.8300222468376159
Epoch 2/20, Loss: 0.2968339815735817
Epoch 3/20, Loss: 0.17964713975787164
Epoch 4/20, Loss: 0.1398733853548765
Epoch 5/20, Loss: 0.11016696263104678
Epoch 6/20, Loss: 0.07548088643699885
Epoch 7/20, Loss: 0.06186456518247724
Epoch 8/20, Loss: 0.04855075437575579
Epoch 9/20, Loss: 0.04036476534791291
Epoch 10/20, Loss: 0.07038516880944372
Epoch 11/20, Loss: 0.040686340564861895
Epoch 12/20, Loss: 0.01774655217304826
Epoch 13/20, Loss: 0.012487249500118196
Epoch 14/20, Loss: 0.009882506728172303
Epoch 15/20, Loss: 0.006876806642394513
Epoch 16/20, Loss: 0.0047618341480847445
Epoch 17/20, Loss: 0.0038311767065897584
Epoch 18/20, Loss: 0.004271073513664305
Epoch 19/20, Loss: 0.00292295178049244
Epoch 20/20, Loss: 0.0027273541502654554


In [50]:
#Validate Model now

model.eval()

correct = 0
total = 0

with torch.no_grad():
  for x,y in test_loader:
    y_pred = model(x)
    _,predicted = torch.max(y_pred.data,1)

    total += y.size(0)
    correct += (predicted == y).sum().item()

print(f"Accuracy: {100*correct/total}%")

Accuracy: 92.5%
